# Dual-Head Heart Disease Model (Binary + Severity) 

In [ ]:
# Imports
from pathlib import Path
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

from sklearn.model_selection import train_test_split, ParameterGrid
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.inspection import permutation_importance
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    confusion_matrix,
    recall_score,
    roc_auc_score,
    average_precision_score
)

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid')

In [ ]:
# Reproducibility Setup
def set_seed(seed=42):
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

SEEDS = [7, 21, 42]
set_seed(42)
print('Seeds:', SEEDS)

## Data Loading, Provenance, and Audit

In [ ]:
# Load and Audit Data
column_names = [
    'age', 'sex', 'cp', 'trestbps', 'chol', 'fbs', 'restecg',
    'thalach', 'exang', 'oldpeak', 'slope', 'ca', 'thal', 'diagnosis'
]

candidate_paths = [
    Path('/home/rohanprashant/Downloads/heart+disease/processed.cleveland.data'),
    Path('/home/rohanprashant/Downloads/heart+disease/cleveland.data')
]

df = None
used_source = None
for p in candidate_paths:
    if p.exists():
        df = pd.read_csv(p, header=None, names=column_names)
        used_source = str(p)
        break

if df is None:
    backup_url = 'https://raw.githubusercontent.com/dataprofessor/data/master/heart-disease-cleveland.csv'
    df = pd.read_csv(backup_url)
    df.columns = [c.strip() for c in df.columns]
    used_source = backup_url

df = df.replace('?', np.nan)
for c in column_names:
    if c in df.columns:
        df[c] = pd.to_numeric(df[c], errors='coerce')

print('Data source:', used_source)
print('Raw shape:', df.shape)
print('Missing by column:')
display(df.isna().sum().to_frame('missing').T)
print('Duplicate rows:', df.duplicated().sum())

df = df.dropna().copy()
df['diagnosis'] = df['diagnosis'].astype(int)
df['diagnosis_binary'] = (df['diagnosis'] > 0).astype(int)

print('Clean shape:', df.shape)
print('Binary distribution:')
display(df['diagnosis_binary'].value_counts().to_frame('count'))
print('Multiclass distribution:')
display(df['diagnosis'].value_counts().sort_index().to_frame('count'))

In [ ]:
# Split and Scale Data
feature_cols = [c for c in df.columns if c not in ['diagnosis', 'diagnosis_binary']]
X = df[feature_cols].values
y_bin = df['diagnosis_binary'].values
y_multi = df['diagnosis'].values

# Stratify by multiclass label to preserve severity distribution
X_train, X_temp, yb_train, yb_temp, ym_train, ym_temp = train_test_split(
    X, y_bin, y_multi, test_size=0.30, random_state=42, stratify=y_multi
)

X_val, X_test, yb_val, yb_test, ym_val, ym_test = train_test_split(
    X_temp, yb_temp, ym_temp, test_size=0.50, random_state=42, stratify=ym_temp
)

print('Train/Val/Test:', len(ym_train), len(ym_val), len(ym_test))

scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_val_s = scaler.transform(X_val)
X_test_s = scaler.transform(X_test)

X_train_t = torch.tensor(X_train_s, dtype=torch.float32)
X_val_t = torch.tensor(X_val_s, dtype=torch.float32)
X_test_t = torch.tensor(X_test_s, dtype=torch.float32)

yb_train_t = torch.tensor(yb_train, dtype=torch.long)
yb_val_t = torch.tensor(yb_val, dtype=torch.long)
yb_test_t = torch.tensor(yb_test, dtype=torch.long)

ym_train_t = torch.tensor(ym_train, dtype=torch.long)
ym_val_t = torch.tensor(ym_val, dtype=torch.long)
ym_test_t = torch.tensor(ym_test, dtype=torch.long)

## Dual-Head Neural Architecture

In [ ]:
# Define Dual-Head Model and Metrics
class DualHeadNet(nn.Module):
    def __init__(self, input_dim, hidden_size, dropout):
        super().__init__()
        self.shared = nn.Sequential(
            nn.Linear(input_dim, hidden_size),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_size, hidden_size // 2),
            nn.ReLU(),
            nn.Dropout(dropout)
        )
        self.bin_head = nn.Linear(hidden_size // 2, 2)
        self.multi_head = nn.Linear(hidden_size // 2, 5)

    def forward(self, x):
        z = self.shared(x)
        return self.bin_head(z), self.multi_head(z)

def train_dual(model, X_tr, yb_tr, ym_tr, X_v, yb_v, ym_v, lr, wd, alpha=1.0, epochs=140, batch_size=32):
    ce = nn.CrossEntropyLoss()
    opt = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=wd)
    loader = DataLoader(TensorDataset(X_tr, yb_tr, ym_tr), batch_size=batch_size, shuffle=True)

    train_losses, val_losses = [], []
    for _ in range(epochs):
        model.train()
        total = 0.0
        for xb, yb, ym in loader:
            opt.zero_grad()
            lb, lm = model(xb)
            loss = ce(lb, yb) + alpha * ce(lm, ym)
            loss.backward()
            opt.step()
            total += loss.item() * len(xb)

        train_losses.append(total / len(X_tr))

        model.eval()
        with torch.no_grad():
            lvb, lvm = model(X_v)
            val_loss = ce(lvb, yb_v) + alpha * ce(lvm, ym_v)
        val_losses.append(val_loss.item())

    return train_losses, val_losses

def predict_dual(model, X):
    model.eval()
    with torch.no_grad():
        lb, lm = model(X)
        pb = torch.softmax(lb, dim=1).cpu().numpy()[:, 1]
        pm = torch.softmax(lm, dim=1).cpu().numpy()
        yb_hat = (pb >= 0.5).astype(int)
        ym_hat = pm.argmax(axis=1)
    return pb, pm, yb_hat, ym_hat

def binary_metrics(y_true, y_prob, threshold=0.5):
    y_pred = (y_prob >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
    sensitivity = tp / (tp + fn) if (tp + fn) > 0 else np.nan
    specificity = tn / (tn + fp) if (tn + fp) > 0 else np.nan
    return {
        'accuracy': accuracy_score(y_true, y_pred),
        'f1': f1_score(y_true, y_pred),
        'sensitivity': sensitivity,
        'specificity': specificity,
        'roc_auc': roc_auc_score(y_true, y_prob),
        'pr_auc': average_precision_score(y_true, y_prob),
        'cm': np.array([[tn, fp], [fn, tp]])
    }

In [ ]:
# Hyperparameter Search
param_grid = {
    'lr': [1e-2, 1e-3],
    'hidden_size': [32, 64, 128],
    'dropout': [0.0, 0.3],
    'wd': [0.0, 1e-4],
    'alpha': [0.8, 1.0]
}

input_dim = X_train_t.shape[1]
search_rows = []

for cfg in ParameterGrid(param_grid):
    seed_scores = []
    for s in SEEDS:
        set_seed(s)
        model = DualHeadNet(input_dim, cfg['hidden_size'], cfg['dropout'])
        train_dual(
            model, X_train_t, yb_train_t, ym_train_t, X_val_t, yb_val_t, ym_val_t,
            lr=cfg['lr'], wd=cfg['wd'], alpha=cfg['alpha']
        )
        pb_val, _, _, ym_val_hat = predict_dual(model, X_val_t)
        bin_auc = roc_auc_score(yb_val, pb_val)
        multi_macro_f1 = f1_score(ym_val, ym_val_hat, average='macro')
        seed_scores.append(0.5 * bin_auc + 0.5 * multi_macro_f1)

    row = dict(cfg)
    row['val_composite_mean'] = float(np.mean(seed_scores))
    row['val_composite_std'] = float(np.std(seed_scores))
    search_rows.append(row)

search_df = pd.DataFrame(search_rows).sort_values('val_composite_mean', ascending=False).reset_index(drop=True)
display(search_df.head(8))
best_cfg = search_df.iloc[0].to_dict()
best_cfg

In [ ]:
# Multi-Seed Training and Selection
seed_rows = []
seed_models = []

for s in SEEDS:
    set_seed(s)
    model = DualHeadNet(input_dim, int(best_cfg['hidden_size']), float(best_cfg['dropout']))
    train_dual(
        model, X_train_t, yb_train_t, ym_train_t, X_val_t, yb_val_t, ym_val_t,
        lr=float(best_cfg['lr']), wd=float(best_cfg['wd']), alpha=float(best_cfg['alpha'])
    )

    pb_test, pm_test, yb_hat, ym_hat = predict_dual(model, X_test_t)

    bm = binary_metrics(yb_test, pb_test, threshold=0.5)
    mm = {
        'multi_accuracy': accuracy_score(ym_test, ym_hat),
        'multi_macro_f1': f1_score(ym_test, ym_hat, average='macro'),
        'multi_weighted_f1': f1_score(ym_test, ym_hat, average='weighted')
    }

    seed_rows.append({
        'seed': s,
        'bin_roc_auc': bm['roc_auc'],
        'bin_f1': bm['f1'],
        'multi_macro_f1': mm['multi_macro_f1'],
        'multi_weighted_f1': mm['multi_weighted_f1']
    })

    seed_models.append({
        'seed': s, 'model': model,
        'pb_test': pb_test, 'pm_test': pm_test,
        'yb_hat': yb_hat, 'ym_hat': ym_hat
    })

seed_df = pd.DataFrame(seed_rows)
display(seed_df)
print('Mean +/- std across seeds')
display(seed_df.drop(columns=['seed']).agg(['mean', 'std']))

rep_seed = int(seed_df.sort_values('bin_roc_auc', ascending=False).iloc[0]['seed'])
rep = [m for m in seed_models if m['seed'] == rep_seed][0]
print('Representative seed:', rep_seed)

In [ ]:
# Evaluate Representative Model
pb_test, pm_test = rep['pb_test'], rep['pm_test']
yb_hat, ym_hat = rep['yb_hat'], rep['ym_hat']

bm = binary_metrics(yb_test, pb_test, threshold=0.5)
mm_acc = accuracy_score(ym_test, ym_hat)
mm_macro_f1 = f1_score(ym_test, ym_hat, average='macro')
mm_weighted_f1 = f1_score(ym_test, ym_hat, average='weighted')
mm_recall_per_class = recall_score(ym_test, ym_hat, average=None, labels=[0,1,2,3,4])

print('Binary head metrics')
for k in ['accuracy', 'f1', 'sensitivity', 'specificity', 'roc_auc', 'pr_auc']:
    print(f'{k:12s}: {bm[k]:.4f}')

print('\nMulticlass head metrics')
print(f'accuracy      : {mm_acc:.4f}')
print(f'macro_f1      : {mm_macro_f1:.4f}')
print(f'weighted_f1   : {mm_weighted_f1:.4f}')
print('per-class recall (0..4):', np.round(mm_recall_per_class, 4))

fig, ax = plt.subplots(1, 2, figsize=(11, 4))
sns.heatmap(bm['cm'], annot=True, fmt='d', cmap='Blues',
            xticklabels=['Pred 0','Pred 1'], yticklabels=['True 0','True 1'], ax=ax[0])
ax[0].set_title('Binary Head Confusion Matrix')

cm_multi = confusion_matrix(ym_test, ym_hat, labels=[0,1,2,3,4])
sns.heatmap(cm_multi, annot=True, fmt='d', cmap='Greens',
            xticklabels=[0,1,2,3,4], yticklabels=[0,1,2,3,4], ax=ax[1])
ax[1].set_title('Multiclass Head Confusion Matrix')

plt.tight_layout()
plt.show()

## Joint Error Analysis



In [ ]:
# Joint Error Analysis
err_df = pd.DataFrame(X_test, columns=feature_cols)
err_df['y_bin_true'] = yb_test
err_df['y_bin_pred'] = yb_hat
err_df['y_multi_true'] = ym_test
err_df['y_multi_pred'] = ym_hat

# Critical healthcare error: disease present but binary predicts absent
critical_fn = err_df[(err_df['y_bin_true'] == 1) & (err_df['y_bin_pred'] == 0)]

# Among true positives, severity mismatch
severity_err = err_df[(err_df['y_bin_true'] == 1) & (err_df['y_bin_pred'] == 1) & (err_df['y_multi_true'] != err_df['y_multi_pred'])]

print('Critical false negatives (binary head):', len(critical_fn))
print('Severity mismatches among binary true positives:', len(severity_err))

if len(critical_fn) > 0:
    display(critical_fn[['age','sex','cp','thalach','oldpeak','y_multi_true','y_multi_pred']].head())

## Classical Baselines for Comparison



In [ ]:
# Classical Baseline Comparison
lr_bin = LogisticRegression(max_iter=2000, random_state=42)
lr_bin.fit(X_train_s, yb_train)
lr_bin_prob = lr_bin.predict_proba(X_test_s)[:, 1]
lr_bin_metrics = binary_metrics(yb_test, lr_bin_prob, threshold=0.5)

lr_multi = LogisticRegression(max_iter=3000, multi_class='multinomial', random_state=42)
lr_multi.fit(X_train_s, ym_train)
lr_multi_pred = lr_multi.predict(X_test_s)

baseline_tbl = pd.DataFrame([
    {
        'model': 'DualHeadNN',
        'bin_roc_auc': bm['roc_auc'],
        'bin_f1': bm['f1'],
        'multi_macro_f1': mm_macro_f1,
        'multi_weighted_f1': mm_weighted_f1
    },
    {
        'model': 'LogReg_Bin+Multi',
        'bin_roc_auc': lr_bin_metrics['roc_auc'],
        'bin_f1': lr_bin_metrics['f1'],
        'multi_macro_f1': f1_score(ym_test, lr_multi_pred, average='macro'),
        'multi_weighted_f1': f1_score(ym_test, lr_multi_pred, average='weighted')
    }
]).set_index('model')
display(baseline_tbl.round(4))

## Interpretability (Permutation Importance)

In [ ]:
# Permutation Importance and Optional SHAP
perm_bin = permutation_importance(
    lr_bin, X_test_s, yb_test, scoring='f1', n_repeats=20, random_state=42
)
perm_multi = permutation_importance(
    lr_multi, X_test_s, ym_test, scoring='f1_macro', n_repeats=20, random_state=42
)

perm_bin_df = pd.DataFrame({
    'feature': feature_cols,
    'importance_mean': perm_bin.importances_mean
}).sort_values('importance_mean', ascending=False)

perm_multi_df = pd.DataFrame({
    'feature': feature_cols,
    'importance_mean': perm_multi.importances_mean
}).sort_values('importance_mean', ascending=False)

fig, ax = plt.subplots(1, 2, figsize=(12, 5))
sns.barplot(data=perm_bin_df.head(8), x='importance_mean', y='feature', color='royalblue', ax=ax[0])
ax[0].set_title('Binary Baseline Importance')
sns.barplot(data=perm_multi_df.head(8), x='importance_mean', y='feature', color='seagreen', ax=ax[1])
ax[1].set_title('Multiclass Baseline Importance')
plt.tight_layout()
plt.show()

print('Optional SHAP for baselines (if available):')
try:
    import shap
    _ = shap.LinearExplainer(lr_bin, X_train_s)
    print('SHAP ready for binary baseline.')
except Exception as e:
    print('SHAP unavailable or failed:', e)

## Fairness, Risks, and Mitigation



In [ ]:
# Subgroup Fairness Check
fair_df = pd.DataFrame(X_test, columns=feature_cols)
fair_df['y_true'] = yb_test
fair_df['y_prob'] = pb_test

def subgroup_bin(df_sub):
    if len(df_sub) < 5 or df_sub['y_true'].nunique() < 2:
        return {'n': len(df_sub), 'sensitivity': np.nan, 'specificity': np.nan, 'f1': np.nan}
    m = binary_metrics(df_sub['y_true'].values, df_sub['y_prob'].values, threshold=0.5)
    return {'n': len(df_sub), 'sensitivity': m['sensitivity'], 'specificity': m['specificity'], 'f1': m['f1']}

rows = []
for sx, g in fair_df.groupby('sex'):
    r = subgroup_bin(g)
    r['group'] = f'sex={int(sx)}'
    rows.append(r)

fair_df['age_group'] = np.where(fair_df['age'] >= 60, 'age>=60', 'age<60')
for ag, g in fair_df.groupby('age_group'):
    r = subgroup_bin(g)
    r['group'] = ag
    rows.append(r)

display(pd.DataFrame(rows)[['group', 'n', 'f1', 'sensitivity', 'specificity']].round(4))